# GETM Dutch Wadden Sea — wave validation against RWS observations, 2015

Compares `dws_500m.3d.2015??.nc` against the Rijkswaterstaat observations in
this folder (`../processed/`).

## What this notebook validates

| Model variable | Observation | Note |
|---|---|---|
| `Hs_out` | `hm0` | significant wave height, direct |
| `Tz_out` | `tm02` | zero-crossing period, direct |
| `elev` | `wl` | water level; **read the aliasing warning below** |
| `EUWIND`/`EVWIND` | KNMI wind | forcing sanity check, not model skill |

## What it cannot validate, and why

**Currents.** There are *no* current-meter observations anywhere in the Dutch
Wadden Sea in 2015. RWS current data starts at Eemshaven in 2020. The `u`/`v`
comparison code is written and ready (§10) but will find nothing until you run
a year ≥ 2020.

**Tidal dynamics, from daily snapshots.** With one instantaneous value per day
at 00:00, the M2 tide (12.42 h) is aliased to a period of

$$\left|f_{M2} - 2f_s\right|^{-1} = |1.9323 - 2|^{-1} \approx 14.8\ \text{days}$$

so daily 00:00 `elev` and `u`/`v` will appear as a spurious ~15-day
oscillation, and a *perfect* tidal model would still score badly against
instantaneous observations. Water level is therefore reported in §9 with that
caveat, and should not be read as tidal skill. Waves are far less affected:
`Hs` is driven by wind on synoptic timescales, so a daily sample is a coarse
but honest sample of the wave climate.

**Statistical independence.** 365 daily samples of a quantity with a 1–2 day
decorrelation time gives on the order of 100–200 effective degrees of freedom,
not 365. Treat confidence intervals accordingly.

## Three model-file details worth knowing

1. Each monthly file carries **32 timesteps** in January — 00:00 on 1 Jan
   through 00:00 on 1 Feb inclusive. Concatenating twelve files therefore
   **duplicates every month boundary**. §5 collapses them.
2. `Hs_out` and `Tz_out` are **`-9998` over the whole domain on
   the first frame of every file**, while `elev` is valid there. `-9998` is not
   the declared `_FillValue` (`-9999`), so xarray does **not** mask it: left
   alone it enters the statistics as a −9998 m wave height. §5 masks it.
3. Taken together with the `averaged = 1, 0` attribute those variables carry
   (and `elev`, `u`, `v` do not), the empty first frame says the wave fields
   are **time-averaged over the interval ending at the timestamp** — so the
   first day of each month is reported on frame 2, exactly as you suspected.
   §6 tests that against the observations rather than assuming it.

---
## 1. Configuration

The only cell you should normally need to edit.

In [ ]:
from pathlib import Path
import sys, warnings, json

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- paths -----------------------------------------------------------------
OBS_ROOT  = Path("/export/lv9/projects/dws/results/validation/waves/")              # the RWS/waves folder
MODEL_DIR = Path("/export/lv9/projects/dws/model_output/archived_runs/effective_fetch/spinup_01/")  # <-- EDIT: folder with the 12 files
MODEL_GLOB = "dws_500m.3d.2015??.nc"
OUT_DIR   = OBS_ROOT / "output" / "effective_fetch/" ; OUT_DIR.mkdir(exist_ok=True)

YEAR = 2015

# --- matching --------------------------------------------------------------
MAX_MATCH_KM  = 3.0    # reject a station further than this from a wet cell

# Nearest-neighbour is one choice among several. A (2R+1)x(2R+1) window is
# extracted around each station so that alternatives can be compared without
# re-reading the files: R=2 is a 5x5 stencil, 2.5 km across at 500 m.
NEIGHBOURHOOD_R = 2
# "nearest" | "depth-matched" | "window-median" - see section 7b before changing
CELL_STRATEGY = "nearest"
MIN_MODEL_DEPTH_M = 0.5
MIN_PAIRS     = 30     # skip station/variable pairs with fewer matched days
MATCH_TOL     = pd.Timedelta("10min")   # obs<->model instantaneous tolerance

# --- observation conditioning ----------------------------------------------
# The wide tables carry qc_* flags and three ready-made validity masks; see
# README section 4b for what each one means and why it exists. With this on,
# an observation the instrument could not actually have measured is blanked
# before anything is computed from it, so every skill score, scatter and map
# below inherits the same conditioning. Set False to compare against the raw
# delivered values.
USE_QC_MASKS = True

# GETM writes -9998 across the whole domain into the time-AVERAGED fields on
# the first frame of every file - there is no preceding interval to average
# over yet. That is NOT the declared _FillValue (-9999), so xarray does not
# mask it and it would otherwise poison every statistic. Treat anything this
# negative as missing.
SENTINEL_BELOW = -9990.0

# model variable -> observation column
PAIRS = {
    "Hs_out":    "hm0",
    "Tz_out":    "tp",
    "elev":      "wl",
}
# Orbital velocity is deliberately absent. It is derived on both sides - from
# hm0/tm02/depth in the observations and from the model's own scheme - so a
# disagreement confounds wave physics with two definitions and a bed level.
# Hs and Tz are measured directly and are the honest comparison.
# variables pulled from the model files (2-D only; keeps extraction cheap)
MODEL_VARS = ["elev", "u", "v", "Hs_out", "Tz_out",
              "TauW", "TauC", "EUWIND", "EVWIND", "dry_z"]
# TauW/TauC are pulled for reference only - no observations exist for them.
STATIC_VARS = ["lonc", "latc", "bathymetry", "convc"]

# --- house style (sequential blue / diverging blue-red, grey midpoint) ------
BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#0d366b"]
SEQ = LinearSegmentedColormap.from_list("seq_blue", BLUE)
DIV = LinearSegmentedColormap.from_list("bl_rd", [
    "#0d366b", "#256abf", "#3987e5", "#86b6ef", "#cde2fb", "#f0efec",
    "#fbd7d5", "#f3a9a4", "#e34948", "#c62f2e", "#96201f"])
C_MOD, C_OBS = "#2a78d6", "#0b0b0b"
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e2e1dd", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 9, "figure.dpi": 110,
})

print("obs root :", OBS_ROOT)
print("model dir:", MODEL_DIR, "->", "FOUND" if MODEL_DIR.is_dir() else "NOT FOUND (edit MODEL_DIR)")

---
## 2. Observations

Reads `processed/csv/<station>.csv.gz` (10-minute, UTC, SI units) and keeps the
stations with data in the target year. See `../README.md` §4 for the quality
control already applied.

## 2b. Observations, conditioned

The RWS wide tables carry six `qc_*` flags and three validity masks alongside the measurements (README section 4b). They exist because parts of this archive are not measurements: peak period saturates at both edges of the 0.03-0.50 Hz band when the sea is flat, `Hm0` is quantised at 0.01 m so anything under ~0.05 m is arithmetic, and the intertidal step gauges keep reporting after the flat has dried.

Applying the masks here, at the point the observations are loaded, means every skill score, scatter panel, Taylor diagram and bias map below is computed on conditioned data without any of them needing to know. Set `USE_QC_MASKS = False` in the configuration cell to see what the raw values give instead - the difference is the measure of how much the conditioning matters for your run.


In [ ]:
# The conditioning step. Each wave column is blanked wherever its own mask
# says the value is not a measurement, and the water level is blanked wherever
# the gauge was sitting dry on an intertidal flat - an emergent step gauge
# reports its lowest wet sensor, which would otherwise read as a large model
# bias in elevation rather than as a missing observation.
MASK_FOR = {"hm0": "valid_hm0", "h13": "valid_hm0", "hmax": "valid_hm0",
            "tm02": "valid_tm02", "tm_10": "valid_tm02", "tm02_hf": "valid_tm02",
            "tm_10_hf": "valid_tm02", "tp": "valid_tp", "fp": "valid_tp"}
EMERGENT_BLANKS = ["wl", "depth", "u_orb", "u_orb_rms"]
qc_removed = []

def apply_qc(d, code_):
    """Blank observations the flags say are not measurements. Nothing is
    dropped from the table - only the affected values become NaN."""
    before = {c: int(d[c].notna().sum()) for c in d.columns
              if c in MASK_FOR or c in EMERGENT_BLANKS}
    for col, mask in MASK_FOR.items():
        if col in d.columns and mask in d.columns:
            d.loc[d[mask] != 1, col] = np.nan
    if "qc_emergent" in d.columns:
        cols = [c for c in EMERGENT_BLANKS if c in d.columns]
        if cols:
            d.loc[d["qc_emergent"] == 1, cols] = np.nan
    for c, n0 in before.items():
        n1 = int(d[c].notna().sum())
        if n0 and n1 < n0:
            qc_removed.append({"station": code_, "column": c, "kept": n1,
                               "was": n0, "removed_pct": 100 * (1 - n1 / n0)})
    return d

ov = pd.read_csv(OBS_ROOT / "processed" / "station_overview.csv")

OBS = {}
for code_ in ov["code"]:
    fp = OBS_ROOT / "processed" / "csv" / f"{code_}.csv.gz"
    if not fp.exists():
        continue
    d = pd.read_csv(fp, index_col=0, parse_dates=True)
    d = d.loc[f"{YEAR}-01-01":f"{YEAR}-12-31"]
    if d.empty or not d.notna().any().any():
        continue
    if USE_QC_MASKS:
        d = apply_qc(d, code_)
    OBS[code_] = d

meta = ov.set_index("code").loc[list(OBS)]
counts = pd.DataFrame({c: {v: int(OBS[c][v].notna().sum()) if v in OBS[c] else 0
                           for v in PAIRS.values()} for c in OBS}).T
counts["lat"] = meta["lat"]; counts["lon"] = meta["lon"]
print(f"{len(OBS)} stations with {YEAR} data\n")
display(counts.sort_values("hm0", ascending=False))

if counts.get("cur_speed", pd.Series(dtype=int)).sum() == 0:
    print("\nNo current observations in this year - the u/v comparison in "
          "section 10 will be empty (expected for 2015).")
if USE_QC_MASKS:
    qc_tab = pd.DataFrame(qc_removed)
    if qc_tab.empty:
        print("\nQC masks on: nothing needed blanking in this year.")
    else:
        piv = (qc_tab.pivot_table(index="station", columns="column",
                                  values="removed_pct", aggfunc="first")
               .round(1).fillna(0.0))
        print("\nQC masks on - per cent of each column blanked as "
              "not-a-measurement (README 4b):")
        display(piv.loc[piv.max(axis=1).sort_values(ascending=False).index])
else:
    print("\nQC masks OFF - comparing against the raw delivered values.")


---
## 3. Model grid

Only the static fields are read here. `bathymetry` is positive-down depth
below the model reference level; land is masked by its `_FillValue`.

**`lonc`/`latc` need repairing first.** They carry `_FillValue = -999`, which
xarray masks to NaN outside the computational domain. That breaks two things:
`pcolormesh` refuses non-finite coordinates outright, and — more quietly — a
KD-tree built on NaN coordinates matches stations to nonsense. Both are fixed
below: the holes are filled by interpolation in index space, and the wet mask
additionally requires finite coordinates.

In [ ]:
files = sorted(MODEL_DIR.glob(MODEL_GLOB))
assert files, f"no model files matching {MODEL_GLOB} in {MODEL_DIR}"
print(f"{len(files)} model files:", ", ".join(f.name for f in files[:3]), "...")

def fill_coord(a, name=""):
    """Fill the -999 holes in lonc/latc.

    A GETM grid is a rotated regular grid, so lon and lat are very nearly
    linear in the index space (i, j). Interior holes are interpolated, and
    anything outside the convex hull (typically the halo ring) is extrapolated
    from a least-squares plane. The residual of that plane on the VALID points
    is printed so you can see whether the linear assumption holds.
    """
    a = np.asarray(np.ma.filled(a, np.nan), dtype=float)
    bad = ~np.isfinite(a)
    if not bad.any():
        return a
    jj, ii = np.mgrid[0:a.shape[0], 0:a.shape[1]]
    good = ~bad
    A = np.column_stack([np.ones(good.sum()), ii[good], jj[good]])
    coef, *_ = np.linalg.lstsq(A, a[good], rcond=None)
    plane = coef[0] + coef[1] * ii + coef[2] * jj
    resid = float(np.abs(a[good] - plane[good]).max())

    out = a.copy()
    try:                                   # interior holes: proper interpolation
        from scipy.interpolate import griddata
        out[bad] = griddata((ii[good], jj[good]), a[good],
                            (ii[bad], jj[bad]), method="linear")
    except Exception:
        pass
    still = ~np.isfinite(out)
    out[still] = plane[still]              # halo / outside hull: extrapolate
    print(f"  {name}: filled {int(bad.sum())} of {a.size} "
          f"({100*bad.mean():.1f}%), plane residual {resid:.4f} deg")
    if resid > 0.05:
        print(f"    ! {name} is not well described by a plane; check the map "
              "below before trusting the filled edges")
    return out


with xr.open_dataset(files[0]) as d0:
    lon2d = fill_coord(d0["lonc"].values, "lonc")
    lat2d = fill_coord(d0["latc"].values, "latc")
    bathy = np.asarray(np.ma.filled(d0["bathymetry"].values, np.nan), float)
    convc = d0["convc"].values if "convc" in d0 else None
    tunits = d0["time"].encoding.get("units", "?")
    print("grid", lon2d.shape, "| time units:", tunits)
    print("model 2-D variables present:",
          [v for v in MODEL_VARS if v in d0.variables])
    missing = [v for v in MODEL_VARS if v not in d0.variables]
    if missing:
        print("NOT in file (will be skipped):", missing)
    # does the header claim these are averages?
    for v in ("Hs_out", "elev"):
        if v in d0.variables:
            print(f"  {v}: attrs = { {k: val for k, val in d0[v].attrs.items() if k in ('averaged','long_name')} }")

# finite coordinates are part of "usable": a KD-tree built on NaN silently
# mismatches every station
wet = (np.isfinite(bathy) & (bathy > MIN_MODEL_DEPTH_M)
       & np.isfinite(lon2d) & np.isfinite(lat2d))
assert wet.any(), "no usable wet cells - check bathymetry and MIN_MODEL_DEPTH_M"
print(f"\nwet cells: {wet.sum():,} of {wet.size:,}")
print(f"grid extent: lon {np.nanmin(lon2d):.3f}..{np.nanmax(lon2d):.3f}, "
      f"lat {np.nanmin(lat2d):.3f}..{np.nanmax(lat2d):.3f}")
if convc is not None:
    print(f"grid rotation convc: {np.nanmin(convc):.2f}..{np.nanmax(convc):.2f} deg "
          "(u/v are labelled 'global x/y direction', i.e. already geographic)")

---
## 4. Station → grid matching

Nearest **wet** cell on a local equirectangular projection. Stations further
than `MAX_MATCH_KM` from any wet cell are outside the domain (or on a flat the
model treats as land) and are dropped.

A tidal-flat station may legitimately sit in a cell that dries; that is a
model–reality difference worth seeing, not a matching error, so drying is not
used as a rejection criterion here.

In [ ]:
R_EARTH = 6371.0
lat0 = float(np.nanmean(lat2d))

def _xy(lon, lat):
    return (np.radians(lon) * np.cos(np.radians(lat0)) * R_EARTH,
            np.radians(lat) * R_EARTH)

jw, iw = np.nonzero(wet)
xw, yw = _xy(lon2d[wet], lat2d[wet])
tree = cKDTree(np.column_stack([xw, yw]))

rows = []
for code_ in OBS:
    slat, slon = float(meta.loc[code_, "lat"]), float(meta.loc[code_, "lon"])
    sx, sy = _xy(slon, slat)
    dist, k = tree.query([sx, sy])
    j, i = int(jw[k]), int(iw[k])
    rows.append({"code": code_, "lat": slat, "lon": slon,
                 "j": j, "i": i, "dist_km": float(dist),
                 "model_lat": float(lat2d[j, i]), "model_lon": float(lon2d[j, i]),
                 "model_depth_m": float(bathy[j, i])})

match = pd.DataFrame(rows).set_index("code").sort_values("dist_km")
match["inside"] = match["dist_km"] <= MAX_MATCH_KM

print(f"{int(match['inside'].sum())} of {len(match)} stations inside the domain "
      f"(<= {MAX_MATCH_KM} km from a wet cell)\n")
display(match[["lat", "lon", "dist_km", "model_depth_m", "inside"]])

dropped = match.index[~match["inside"]].tolist()
if dropped:
    print("\ndropped (outside domain):", ", ".join(dropped))
STN = match.index[match["inside"]].tolist()

# --- the neighbourhood stencil around each matched cell --------------------
R = NEIGHBOURHOOD_R
OFFSETS = [(dj, di) for dj in range(-R, R + 1) for di in range(-R, R + 1)]
CENTRE = OFFSETS.index((0, 0))
ny, nx = lon2d.shape
_mj = match.loc[STN, "j"].to_numpy()[:, None]
_mi = match.loc[STN, "i"].to_numpy()[:, None]
_dj = np.array([o[0] for o in OFFSETS])[None, :]
_di = np.array([o[1] for o in OFFSETS])[None, :]
_J, _I = _mj + _dj, _mi + _di
_in = (_J >= 0) & (_J < ny) & (_I >= 0) & (_I < nx)
WIN_J, WIN_I = np.clip(_J, 0, ny - 1), np.clip(_I, 0, nx - 1)
WIN_OK = _in & wet[WIN_J, WIN_I]                 # usable cells in each window
WIN_DEPTH = np.where(WIN_OK, bathy[WIN_J, WIN_I], np.nan)

print(f"\nneighbourhood: {2*R+1}x{2*R+1} cells "
      f"(~{(2*R+1)*0.5:.1f} km across at 500 m resolution)")
print(f"usable cells per station: min {WIN_OK.sum(axis=1).min()}, "
      f"median {int(np.median(WIN_OK.sum(axis=1)))}, "
      f"max {WIN_OK.sum(axis=1).max()} of {len(OFFSETS)}")
_spread = np.nanmax(WIN_DEPTH, axis=1) - np.nanmin(WIN_DEPTH, axis=1)
print(f"model depth range within the window: median {np.nanmedian(_spread):.1f} m, "
      f"max {np.nanmax(_spread):.1f} m "
      "- the larger this is, the more the nearest-cell choice matters")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.6))
pm = ax.pcolormesh(lon2d, lat2d, np.where(wet, bathy, np.nan),
                   cmap=SEQ, shading="auto")
cb = fig.colorbar(pm, ax=ax, pad=0.012, fraction=0.03)
cb.set_label("model depth below reference (m)"); cb.outline.set_visible(False)

ins = match[match["inside"]]
ax.scatter(ins["lon"], ins["lat"], s=48, facecolor="#e34948",
           edgecolor="white", linewidth=1.2, zorder=3, label="matched station")
if dropped:
    out = match[~match["inside"]]
    ax.scatter(out["lon"], out["lat"], s=44, marker="x", color=INK,
               linewidth=1.4, zorder=3, label="outside domain")
for c_, r_ in ins.iterrows():
    ax.annotate(c_, (r_["lon"], r_["lat"]), fontsize=6.2, color=INK2,
                xytext=(0, 6), textcoords="offset points", ha="center")
ax.set_xlabel("longitude (°E)"); ax.set_ylabel("latitude (°N)")
ax.set_aspect(1 / np.cos(np.deg2rad(lat0)))
ax.set_title("Model bathymetry and matched observation stations", loc="left")
ax.legend(loc="lower right", frameon=True, facecolor=SURFACE, edgecolor=GRID)
fig.tight_layout(); fig.savefig(OUT_DIR / "fig1_domain.png", dpi=160)

---
## 5. Extract model time series at the stations

One pass per monthly file, indexing only the matched cells — no `dask`
required and memory stays small.

Two file quirks are handled here.

**The `-9998` first frame.** `Hs_out` and `Tz_out` are `-9998`
over the entire domain on the first frame of every file, while `elev` is valid
there. `-9998` is *not* the declared `_FillValue` (`-9999`), so xarray will not
mask it — left alone it enters the statistics as a −9998 m wave height. It is
masked explicitly.

**Duplicate month boundaries, which are not equivalent copies.** 00:00 on the
1st appears in two files. In the later file that is the empty first frame of
the averaged variables; in the earlier file it is a valid value. So the
deduplication keeps, per timestamp, whichever copy carries more valid data
rather than simply the first.

In [ ]:
CACHE = OUT_DIR / f"model_at_stations_{YEAR}.nc"

def mask_sentinels(ds):
    """-9998 / -9999 / -99999 are all 'no data' here; only -9999 is declared."""
    for v in ds.data_vars:
        if ds[v].dtype.kind == "f":
            ds[v] = ds[v].where(ds[v] > SENTINEL_BELOW)
    return ds


def extract(files, jj, ii, station_names):
    """Pull the whole (station, win) stencil in one pass over the files."""
    ok = xr.DataArray(WIN_OK, dims=("station", "win"))
    frames = []
    for f in files:
        with xr.open_dataset(f) as d:
            keep = [v for v in MODEL_VARS + STATIC_VARS if v in d.variables]
            pt = d[keep].isel(
                yc=xr.DataArray(jj, dims=("station", "win")),
                xc=xr.DataArray(ii, dims=("station", "win"))).load()
        pt = mask_sentinels(pt).where(ok)     # blank land/out-of-grid stencil
        dead = {v: int(pt[v].isnull().all(dim=["station", "win"]).sum())
                for v in pt.data_vars if "time" in pt[v].dims}
        dead = {k: n for k, n in dead.items() if n}
        print(f"  {f.name}: {pt.sizes.get('time', 0)} steps"
              + (f"  | all-missing frames: {dead}" if dead else ""), flush=True)
        frames.append(pt)

    out = xr.concat(frames, dim="time")
    out = out.assign_coords(station=("station", station_names))
    # lonc/latc/bathymetry/convc are static; concat gave them a pointless time
    # axis that just inflates the cache file
    for v in STATIC_VARS:
        if v in out and "time" in out[v].dims:
            out[v] = out[v].isel(time=0, drop=True)

    # Month boundaries repeat 00:00 of the 1st. The two copies are NOT
    # equivalent: in the later file that frame is the averaged fields' empty
    # first frame. So keep, for each timestamp, the copy carrying the most
    # valid data rather than simply the first.
    score = np.zeros(out.sizes["time"])
    for v in out.data_vars:
        if "time" in out[v].dims:
            other = [d for d in out[v].dims if d != "time"]
            score += out[v].notnull().sum(dim=other).values
    t = out["time"].values
    order = np.lexsort((-score, t))            # time ascending, score descending
    ts = t[order]
    first = np.ones(len(ts), bool)
    first[1:] = ts[1:] != ts[:-1]
    dup = len(t) - int(first.sum())
    out = out.isel(time=order[first]).sortby("time")
    print(f"\n{len(t)} steps read, {dup} duplicate timestamps collapsed "
          f"(keeping the more complete copy), {out.sizes['time']} kept")
    return out

MODW = None
if CACHE.exists():
    cached = xr.open_dataset(CACHE)
    # a cache written before the neighbourhood stencil existed, or with a
    # different radius / station set, cannot be reused
    good = (cached.sizes.get("win") == len(OFFSETS)
            and cached.sizes.get("station") == len(STN)
            and list(map(str, cached["station"].values)) == list(STN))
    if good:
        MODW = cached
        print("loaded cached extraction:", CACHE.name)
    else:
        cached.close()
        print(f"cache {CACHE.name} does not match the current stencil "
              f"(win={cached.sizes.get('win')}, need {len(OFFSETS)}) "
              "- re-extracting")
if MODW is None:
    MODW = extract(files, WIN_J, WIN_I, STN)
    MODW.to_netcdf(CACHE)
    print("cached ->", CACHE)


def select_cell(modw, how, target_depth=None):
    """Reduce the (station, win) stencil to one series per station."""
    if how == "nearest":
        return modw.isel(win=CENTRE)
    if how == "window-median":
        return modw.median(dim="win", skipna=True, keep_attrs=True)
    if how == "depth-matched":
        if target_depth is None:
            raise ValueError("depth-matched needs a target depth per station")
        diff = np.abs(WIN_DEPTH - np.asarray(target_depth, float)[:, None])
        diff = np.where(np.isfinite(diff), diff, np.inf)
        k = np.argmin(diff, axis=1)
        k = np.where(np.isinf(diff.min(axis=1)), CENTRE, k)   # no usable cell
        return modw.isel(win=xr.DataArray(k, dims="station"))
    raise ValueError(f"unknown strategy {how!r}")


MOD = select_cell(MODW, CELL_STRATEGY)
mtimes = pd.DatetimeIndex(MOD["time"].values)
print(f"\ncell strategy: {CELL_STRATEGY!r}  (compared in section 7b)")
print(f"model time: {mtimes[0]} .. {mtimes[-1]}  (n={len(mtimes)})")
print("step spacing:", pd.Series(mtimes).diff().value_counts().head().to_dict())

In [ ]:
# Which variables actually carry data, and where are the holes?
cov = {}
for v in MOD.data_vars:
    if "time" not in MOD[v].dims:
        continue
    ok = MOD[v].notnull().any(dim="station").values
    cov[v] = {"frames_with_data": int(ok.sum()),
              "frames_empty": int((~ok).sum()),
              "first_valid": str(mtimes[ok][0])[:16] if ok.any() else "-",
              "last_valid": str(mtimes[ok][-1])[:16] if ok.any() else "-"}
cov = pd.DataFrame(cov).T
display(cov)

waves = [v for v in ("Hs_out", "Tz_out") if v in cov.index]
if waves and "elev" in cov.index:
    lag = (pd.Timestamp(cov.loc[waves[0], "first_valid"])
           - pd.Timestamp(cov.loc["elev", "first_valid"]))
    print(f"\nWave fields start {lag} after elev.")
    if lag >= pd.Timedelta("1D"):
        print("Consistent with an averaged field: nothing to report until one "
              "averaging interval has elapsed. Expect 'mean of previous 24 h' "
              "to win in section 6.")

---
## 6. Is the model output instantaneous or a daily average?

This matters: an instantaneous 00:00 value must be compared against the
observation *at* 00:00, whereas a daily mean must be compared against a daily
mean of the observations. Getting it wrong inflates the scatter and looks like
model error.

Two pieces of evidence already point the same way, before any fitting:

1. `Hs_out`, `Tz_out`, `TauW`, `TauC` carry an `averaged = 1, 0`
   attribute in the header; `elev`, `u`, `v` do not.
2. Those same variables are empty (`-9998`) on the **first frame of every
   file**, while `elev` is valid there. An averaged field has nothing to report
   until one averaging interval has elapsed — so the value stamped at time *t*
   is the mean over the interval **ending** at *t*, and the first day of each
   file therefore appears on frame 2.

If that reading is right, `mean of previous 24 h` should win below. The four
conventions are still fitted rather than assumed, because the conclusion
changes every number in this notebook.

In [ ]:
def window_mean(series, times, lo_h, hi_h, min_frac=0.5):
    """Mean of `series` over (t+lo_h, t+hi_h] for each t, else NaN."""
    s = series.dropna()
    if s.empty:
        return pd.Series(np.nan, index=times)
    need = min_frac * (hi_h - lo_h) * 6          # 6 samples per hour
    vals = []
    for t in times:
        w = s.loc[t + pd.Timedelta(hours=lo_h): t + pd.Timedelta(hours=hi_h)]
        vals.append(w.mean() if len(w) >= need else np.nan)
    return pd.Series(vals, index=times)

def obs_at(series, times, how):
    if how == "instantaneous":
        s_ = series.dropna()
        s_ = s_[~s_.index.duplicated(keep="first")].sort_index()
        return s_.reindex(times, method="nearest", tolerance=MATCH_TOL)
    if how == "mean of previous 24 h":
        return window_mean(series, times, -24, 0)
    if how == "mean of following 24 h":
        return window_mean(series, times, 0, 24)
    if how == "centred 24 h mean":
        return window_mean(series, times, -12, 12)
    raise ValueError(how)

CONVENTIONS = ["instantaneous", "mean of previous 24 h",
               "mean of following 24 h", "centred 24 h mean"]

test_var, test_obs = "Hs_out", "hm0"
res = []
for how in CONVENTIONS:
    o_all, m_all = [], []
    for c_ in STN:
        if test_obs not in OBS[c_]:
            continue
        o = obs_at(OBS[c_][test_obs], mtimes, how)
        m = pd.Series(MOD[test_var].sel(station=c_).values, index=mtimes)
        ok = o.notna() & m.notna()
        if ok.sum() >= MIN_PAIRS:
            o_all.append(o[ok]); m_all.append(m[ok])
    if not o_all:
        continue
    o = pd.concat(o_all); m = pd.concat(m_all)
    res.append({"convention": how, "n": len(o),
                "r": float(np.corrcoef(o, m)[0, 1]),
                "rmse": float(np.sqrt(((m - o) ** 2).mean())),
                "bias": float(m.mean() - o.mean()),
                "obs_std": float(o.std())})

conv = pd.DataFrame(res).set_index("convention")
display(conv.round(4))

BEST = conv["rmse"].idxmin()
PRIOR = "mean of previous 24 h"     # what the empty first frame implies
MARGIN = 0.05                       # 5% of RMSE

print(f"\nLowest RMSE: '{BEST}' ({conv.loc[BEST, 'rmse']:.3f} m, "
      f"r {conv.loc[BEST, 'r']:.3f})")
spread = conv["rmse"].max() / conv["rmse"].min() - 1
print(f"Spread across all four conventions: {100 * spread:.1f}% of RMSE.")

if PRIOR in conv.index and BEST != PRIOR:
    gain = 1 - conv.loc[BEST, "rmse"] / conv.loc[PRIOR, "rmse"]
    if gain < MARGIN:
        print(f"\n'{BEST}' beats '{PRIOR}' by only {100 * gain:.1f}% - well "
              "within noise for a quantity this smooth, so this test cannot "
              f"separate them. Using '{PRIOR}', which is what the empty first "
              "frame of each file implies physically.")
        CHOSEN = PRIOR
    else:
        print(f"\n'{BEST}' beats '{PRIOR}' by {100 * gain:.1f}% - a real "
              "margin, so the data overrule the first-frame argument. Check "
              "how your GETM output module defines its averaging window.")
        CHOSEN = BEST
else:
    CHOSEN = BEST

# instantaneous is right for elev/u/v (no `averaged` attribute in the header)
OBS_CONVENTION = {v: CHOSEN for v in ("Hs_out", "Tz_out")}
OBS_CONVENTION["elev"] = "instantaneous"
print("\nusing:", OBS_CONVENTION)

---
## 7. Skill metrics

Standard set for wave-model validation:

- **bias** = $\overline{m}-\overline{o}$
- **RMSE**, and **unbiased RMSE** (centred; the part not explained by bias)
- **r** Pearson correlation
- **SI** scatter index, RMSE normalised by the observed mean
- **d** Willmott index of agreement (1 = perfect)
- **slope** of a least-squares fit of model on observation, and the ratio of
  standard deviations (< 1 = model under-varies)

In [ ]:
def skill(o, m):
    o = np.asarray(o, float); m = np.asarray(m, float)
    ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    n = o.size
    if n < MIN_PAIRS:
        return {k: np.nan for k in ("n", "obs_mean", "mod_mean", "bias", "rmse",
                                    "urmse", "mae", "r", "si", "d", "slope",
                                    "std_ratio")} | {"n": n}
    bias = m.mean() - o.mean()
    rmse = np.sqrt(((m - o) ** 2).mean())
    urmse = np.sqrt((((m - m.mean()) - (o - o.mean())) ** 2).mean())
    denom = (np.abs(m - o.mean()) + np.abs(o - o.mean())) ** 2
    return {"n": n, "obs_mean": o.mean(), "mod_mean": m.mean(), "bias": bias,
            "rmse": rmse, "urmse": urmse, "mae": np.abs(m - o).mean(),
            "r": np.corrcoef(o, m)[0, 1],
            "si": rmse / o.mean() if o.mean() else np.nan,
            "d": 1 - ((m - o) ** 2).sum() / denom.sum() if denom.sum() else np.nan,
            "slope": np.polyfit(o, m, 1)[0],
            "std_ratio": m.std() / o.std() if o.std() else np.nan}


def paired(mod_var, obs_col, how=None):
    """{station: DataFrame(obs, mod)} on the model time axis."""
    how = how or OBS_CONVENTION.get(mod_var, "instantaneous")
    out = {}
    if mod_var not in MOD:
        return out
    for c_ in STN:
        if obs_col not in OBS[c_]:
            continue
        o = obs_at(OBS[c_][obs_col], mtimes, how)
        m = pd.Series(MOD[mod_var].sel(station=c_).values, index=mtimes)
        df = pd.DataFrame({"obs": o, "mod": m}).dropna()
        if len(df) >= MIN_PAIRS:
            out[c_] = df
    return out

PAIRED = {mv: paired(mv, oc) for mv, oc in PAIRS.items()}
for mv, d in PAIRED.items():
    print(f"{mv:<10} vs {PAIRS[mv]:<6}: {len(d)} stations, "
          f"{sum(len(v) for v in d.values())} paired days")

In [ ]:
tables = {}
for mv, per_station in PAIRED.items():
    if not per_station:
        continue
    rows = {c_: skill(df["obs"], df["mod"]) for c_, df in per_station.items()}
    t = pd.DataFrame(rows).T.sort_values("rmse")
    allo = pd.concat([df["obs"] for df in per_station.values()])
    allm = pd.concat([df["mod"] for df in per_station.values()])
    t.loc["** ALL **"] = skill(allo, allm)
    tables[mv] = t
    unit = {"Hs_out": "m", "Tz_out": "s", "elev": "m"}[mv]
    print(f"\n=== {mv} vs {PAIRS[mv]}  [{unit}] "
          f"({OBS_CONVENTION.get(mv,'instantaneous')}) ===")
    display(t[["n", "obs_mean", "mod_mean", "bias", "rmse", "urmse",
               "r", "si", "d", "slope", "std_ratio"]].round(3))

---
## 7b. Is the bias a cell-selection artefact, or the model?

Nearest-neighbour matching is a guess. At 500 m a station can sit in a cell
that is hydrodynamically unlike its real surroundings — a channel cell instead
of a flat, or vice versa. Before blaming the wave physics, find out whether a
different cell within a couple of kilometres would have agreed with the
observation.

**The reachable-bias test.** For every cell in the `(2R+1)x(2R+1)` window,
compute the mean bias against the same observation. That gives the range of
biases the neighbourhood could have produced. Then ask one question:

> does that range contain zero?

- **Yes** → some cell within 2.5 km would have had no mean bias at all, so this
  is a *representativeness* problem and the cell choice is worth arguing about.
- **No** → every cell in the neighbourhood errs in the same direction. No
  matching rule can fix it; the discrepancy belongs to the model or its forcing.

This compares *bias to bias*. An earlier version asked how often the
observation fell inside the model's min–max envelope, which turned out to
measure how **narrow** the envelope is rather than anything about cell choice:
offshore the wave field is smooth, the envelope is a few centimetres wide, and
even a near-perfect station scored 1%. Comparing like with like avoids that.

**Three selection rules are compared**, all decided independently of skill:

| rule | idea | risk |
|---|---|---|
| `nearest` | closest wet cell | may land in the wrong feature |
| `depth-matched` | cell in the window whose depth is closest to the station's | only as good as the station depth you feed it |
| `window-median` | median over the window | smooths away real gradients, but is the least arbitrary single number |

> **Do not pick the cell that maximises skill.** With 25 cells to choose from
> you can make almost any model look good; that is calibration wearing
> validation's clothes. Choose the rule *a priori* and report it.

A caution on `depth-matched`: the station depths available here come from the
EMODnet DTM, which `../README.md` §4 flags as unreliable on exactly the tidal
flats where the bias is worst. Matching to a wrong target depth is worse than
not matching at all. Real instrument depths from RWS, or the Dutch
*vaklodingen* surveys, would make this rule trustworthy.

In [ ]:
def skill_for(modsel, mod_var="Hs_out", obs_col="hm0"):
    rows = {}
    how = OBS_CONVENTION.get(mod_var, "instantaneous")
    for c_ in STN:
        if obs_col not in OBS[c_]:
            continue
        o = obs_at(OBS[c_][obs_col], mtimes, how)
        m = pd.Series(modsel[mod_var].sel(station=c_).values, index=mtimes)
        df = pd.DataFrame({"obs": o, "mod": m}).dropna()
        if len(df) >= MIN_PAIRS:
            rows[c_] = skill(df["obs"], df["mod"])
    return pd.DataFrame(rows).T


# station depth target: -bed level (NAP) from the observation catalogue
_bed = ov.set_index("code").reindex(STN)["bed_level_nap_m"].to_numpy(float)
TARGET_DEPTH = -_bed

strategies = {"nearest": select_cell(MODW, "nearest"),
              "depth-matched": select_cell(MODW, "depth-matched",
                                           target_depth=TARGET_DEPTH),
              "window-median": select_cell(MODW, "window-median")}

summary = {}
per_strategy = {}
for name, sel_ in strategies.items():
    t = skill_for(sel_)
    per_strategy[name] = t
    # pooled over stations, weighted by pair count
    w = t["n"] / t["n"].sum()
    summary[name] = {"stations": len(t),
                     "mean |bias|": float((t["bias"].abs() * w).sum()),
                     "mean bias": float((t["bias"] * w).sum()),
                     "mean RMSE": float((t["rmse"] * w).sum()),
                     "mean r": float((t["r"] * w).sum()),
                     "median slope": float(t["slope"].median())}
display(pd.DataFrame(summary).T.round(3))

In [ ]:
# --- the reachable-bias test, every station --------------------------------
def reachable_bias(mod_var="Hs_out", obs_col="hm0"):
    how = OBS_CONVENTION.get(mod_var, "instantaneous")
    field = MODW[mod_var]                       # (time, station, win)
    nwin = field.sizes["win"]
    rows, percell = [], {}
    for c_ in STN:
        if obs_col not in OBS[c_]:
            continue
        o = obs_at(OBS[c_][obs_col], mtimes, how)
        if o.notna().sum() < MIN_PAIRS:
            continue
        cell = field.sel(station=c_).values      # (time, win)
        b = np.full(nwin, np.nan)
        for w in range(nwin):
            pair = pd.DataFrame({"o": o, "m": pd.Series(cell[:, w],
                                                        index=mtimes)}).dropna()
            if len(pair) >= MIN_PAIRS:
                b[w] = pair["m"].mean() - pair["o"].mean()
        if not np.isfinite(b).any():
            continue
        percell[c_] = b
        lo_, hi_ = float(np.nanmin(b)), float(np.nanmax(b))
        rows.append({"code": c_,
                     "n_cells": int(np.isfinite(b).sum()),
                     "obs_mean": float(o.dropna().mean()),
                     "bias_nearest": float(b[CENTRE]),
                     "bias_min": lo_, "bias_max": hi_,
                     "bias_best": float(b[np.nanargmin(np.abs(b))]),
                     "reaches_zero": bool(lo_ <= 0 <= hi_)})
    tab = pd.DataFrame(rows).set_index("code")
    # "how much of the nearest-cell bias a better cell could remove" is only
    # meaningful when there is a bias to remove
    tab["removable"] = np.where(
        tab["bias_nearest"].abs() > 0.05,
        1 - tab["bias_best"].abs() / tab["bias_nearest"].abs().replace(0, np.nan),
        np.nan)
    return tab.sort_values("bias_nearest", key=abs, ascending=False), percell


REACH, PERCELL = reachable_bias()
display(REACH.round(3))

nz = int(REACH["reaches_zero"].sum())
print(f"\n{nz} of {len(REACH)} stations have SOME cell in the "
      f"{2*NEIGHBOURHOOD_R+1}x{2*NEIGHBOURHOOD_R+1} window that would remove "
      "the mean bias entirely - for those, cell choice is a real explanation.")
stuck = REACH[~REACH["reaches_zero"]]
if len(stuck):
    print(f"\n{len(stuck)} stations cannot be fixed by ANY cell. Their smallest "
          f"achievable |bias| ranges {stuck['bias_best'].abs().min():.2f} to "
          f"{stuck['bias_best'].abs().max():.2f} m:")
    print("   " + ", ".join(stuck.index[:12]))
    print("Those are model or forcing errors, not matching errors.")

In [ ]:
# --- one row per station: what bias the neighbourhood could have produced ---
t = REACH.iloc[::-1]                       # largest |bias| ends up at the top
y = np.arange(len(t))
fig, ax = plt.subplots(figsize=(11.5, 0.34 * len(t) + 2.2))
ax.axvline(0, color=INK, lw=1.4, zorder=1)
ax.hlines(y, t["bias_min"], t["bias_max"], color=C_MOD, alpha=0.30, lw=7,
          zorder=2, label=f"range over the {2*NEIGHBOURHOOD_R+1}x"
                          f"{2*NEIGHBOURHOOD_R+1} window")
ax.scatter(t["bias_nearest"], y, s=46, color=C_MOD, edgecolor="white",
           linewidth=1.1, zorder=4, label="nearest cell (what is used)")
ax.scatter(t["bias_best"], y, s=52, marker="|", color=INK, zorder=5,
           label="best cell in the window")
ax.set_yticks(y)
ax.set_yticklabels(t.index, fontsize=8)
ax.set_xlabel("Hs bias, model − observed (m)")
ax.set_title("Could a different model cell have removed the bias?\n"
             "Where the bar crosses zero it could; where it does not, no cell "
             "within 2.5 km agrees with the observation.",
             loc="left", fontsize=11)
for yi, (_, r_) in zip(y, t.iterrows()):
    if not r_["reaches_zero"]:
        # sit the label just beyond the far end of the bar, whichever side
        # of zero the bar is on
        pos, side = ((r_["bias_max"], 7) if r_["bias_min"] > 0
                     else (r_["bias_min"], -7))
        ax.annotate("no cell fixes this", (pos, yi), fontsize=7,
                    color="#96201f", va="center",
                    ha="left" if side > 0 else "right",
                    xytext=(side, 0), textcoords="offset points")
leg = ax.legend(fontsize=8.5, frameon=False, loc="lower right")
for txt in leg.get_texts():
    txt.set_color(INK2)
ax.margins(y=0.01, x=0.12)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig9_reachable_bias.png", dpi=150)
plt.close(fig)

REACH.round(4).to_csv(OUT_DIR / f"cell_reachable_bias_{YEAR}.csv")
pd.DataFrame(summary).T.round(4).to_csv(OUT_DIR / f"cell_strategy_{YEAR}.csv")
print("-> fig9_reachable_bias.png, cell_reachable_bias.csv, cell_strategy.csv")

### Does the cell chosen for Hs also help Tz?

Picking the cell that minimises the Hs bias is only defensible if that cell is
genuinely a better representation of the site. There is a clean way to find
out: choose the cell on **Hs**, then check what it does to **Tz**, which had no
say in the choice.

- Tz improves too, and the two variables tend to prefer the same cell →
  the nearest-neighbour cell really was misplaced, and a better one exists.
- Tz gets no better, or worse, and the preferred cells are unrelated →
  minimising the Hs bias is fitting noise. Keep `nearest` and treat the bias as
  a model error.

This is the difference between finding a better representation of the station
and tuning 25 free parameters against one variable.

In [ ]:
# --- choose the cell on Hs, judge it on Tz ---------------------------------
def cross_check(primary=("Hs_out", "hm0"), secondary=("Tz_out", "tp")):
    pv, po = primary
    sv, so = secondary
    if sv not in MODW or not PERCELL:
        print(f"{sv} not available - skipping the cross-check")
        return None
    how_s = OBS_CONVENTION.get(sv, "instantaneous")
    dj = np.array([o[0] for o in OFFSETS])
    di = np.array([o[1] for o in OFFSETS])
    rows = []
    for c_ in STN:
        if c_ not in PERCELL or so not in OBS[c_]:
            continue
        bp = PERCELL[c_]                       # primary (Hs) bias per cell
        os_ = obs_at(OBS[c_][so], mtimes, how_s)
        if os_.notna().sum() < MIN_PAIRS:
            continue
        sec = MODW[sv].sel(station=c_).values  # (time, win)
        bs = np.full(sec.shape[1], np.nan)     # secondary (Tz) bias per cell
        for w in range(sec.shape[1]):
            pair = pd.DataFrame({"o": os_,
                                 "m": pd.Series(sec[:, w], index=mtimes)}).dropna()
            if len(pair) >= MIN_PAIRS:
                bs[w] = pair["m"].mean() - pair["o"].mean()
        if not (np.isfinite(bp).any() and np.isfinite(bs).any()):
            continue
        w_hs = int(np.nanargmin(np.abs(bp)))   # cell chosen on Hs
        w_tz = int(np.nanargmin(np.abs(bs)))   # cell Tz would have chosen
        if not np.isfinite(bs[w_hs]):
            continue
        rows.append({
            "code": c_,
            "tz_bias_nearest": float(bs[CENTRE]),
            "tz_bias_at_hs_cell": float(bs[w_hs]),
            "tz_bias_best": float(bs[w_tz]),
            "tz_improved": abs(bs[w_hs]) < abs(bs[CENTRE]),
            "same_cell": w_hs == w_tz,
            "cell_gap_km": 0.5 * float(np.hypot(dj[w_hs] - dj[w_tz],
                                                di[w_hs] - di[w_tz])),
            "hs_cell_offset_km": 0.5 * float(np.hypot(dj[w_hs], di[w_hs])),
        })
    return pd.DataFrame(rows).set_index("code") if rows else None


CROSS = cross_check()
if CROSS is not None:
    display(CROSS.round(3))
    n = len(CROSS)
    imp = int(CROSS["tz_improved"].sum())
    same = int(CROSS["same_cell"].sum())
    gain = (CROSS["tz_bias_nearest"].abs()
            - CROSS["tz_bias_at_hs_cell"].abs()).median()
    print(f"\nTz improves at {imp} of {n} stations when the cell is chosen on "
          f"Hs (median change in |Tz bias| {gain:+.3f} s).")
    print(f"Hs and Tz prefer the SAME cell at {same} of {n} stations; "
          f"median distance between their preferred cells "
          f"{CROSS['cell_gap_km'].median():.1f} km.")
    if imp > 0.6 * n and same > 0.3 * n:
        print("\nThe choice carries over to a variable that did not influence "
              "it, so it looks like a genuine representativeness fix rather "
              "than curve-fitting.")
    else:
        print("\nThe Hs-optimal cell does NOT systematically help Tz, and the "
              "two variables disagree about which cell is best. That is the "
              "signature of fitting noise: keep CELL_STRATEGY='nearest' and "
              "treat the Hs bias as a model error.")
    CROSS.round(4).to_csv(OUT_DIR / f"cell_cross_check_{YEAR}.csv")

In [ ]:
if CROSS is not None:
    fig, ax = plt.subplots(figsize=(6.0, 5.8))
    x = CROSS["tz_bias_nearest"].abs()
    y = CROSS["tz_bias_at_hs_cell"].abs()
    lim = [0, float(max(x.max(), y.max())) * 1.08]
    ax.fill_between(lim, lim, lim[1], color=C_MOD, alpha=0.07, linewidth=0)
    ax.plot(lim, lim, "--", color=INK, lw=1.2, zorder=2)
    ax.scatter(x, y, s=55, color=C_MOD, edgecolor="white", linewidth=1.1,
               zorder=3)
    for c_ in CROSS.index:
        ax.annotate(c_, (x[c_], y[c_]), fontsize=6, color=INK2,
                    xytext=(0, 7), textcoords="offset points", ha="center")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.set_xlabel("|Tz bias| at the nearest cell (s)")
    ax.set_ylabel("|Tz bias| at the cell chosen on Hs (s)")
    ax.set_title("Does the Hs-optimal cell also help Tz?\n"
                 "Below the line = yes; shaded above the line = it made Tz "
                 "worse", loc="left", fontsize=10)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fig10_cross_check_tz.png", dpi=160)
    plt.close(fig)
    print("-> fig10_cross_check_tz.png, cell_cross_check.csv")

---
## 8. Figures

In [ ]:
UNITS = {"Hs_out": "m", "Tz_out": "s", "elev": "m"}
LABEL = {"Hs_out": "Hs", "Tz_out": "Tz", "elev": "water level"}
PER_PAGE = 8          # stations per time-series page


def timeseries_pages(mod_var, obs_col, tag=None, per_page=PER_PAGE,
                     pairs=None, note=""):
    """One panel per station, every station, paginated so pages stay legible."""
    pairs = pairs if pairs is not None else PAIRED.get(mod_var, {})
    if not pairs:
        print(f"no pairs for {mod_var}")
        return []
    stns = sorted(pairs, key=lambda c_: -len(pairs[c_]))
    unit, lab = UNITS.get(mod_var, ""), LABEL.get(mod_var, mod_var)
    npages = int(np.ceil(len(stns) / per_page))
    paths = []
    for p in range(npages):
        chunk = stns[p * per_page:(p + 1) * per_page]
        fig, axes = plt.subplots(len(chunk), 1,
                                 figsize=(13, 1.6 * len(chunk)), sharex=True)
        axes = np.atleast_1d(axes)
        for ax, c_ in zip(axes, chunk):
            df = pairs[c_]
            # Observations: put them back on a regular grid so that gaps become
            # NaN and matplotlib BREAKS the line, instead of drawing a straight
            # segment across weeks of missing record.
            if obs_col and obs_col in OBS[c_]:
                raw = OBS[c_][obs_col].dropna()
                if len(raw):
                    raw = raw.reindex(pd.date_range(raw.index.min(),
                                                    raw.index.max(),
                                                    freq="10min"))
                    ax.plot(raw.index, raw.values, color=GRID, linewidth=0.6,
                            zorder=1, label="observed (10-min)")
            # Model: draw the WHOLE model series, not only the days that happen
            # to have an observation - the model exists on those days too.
            if mod_var in MOD:
                m_full = pd.Series(MOD[mod_var].sel(station=c_).values,
                                   index=mtimes)
            else:
                m_full = df["mod"].reindex(mtimes)
            ax.plot(m_full.index, m_full.values, color=C_MOD, linewidth=1.4,
                    zorder=3, label="GETM")
            # Matched observations on the model axis; NaN elsewhere, so this
            # line breaks wherever the comparison has no data.
            o_full = df["obs"].reindex(mtimes)
            ax.plot(o_full.index, o_full.values, color=C_OBS, linewidth=1.1,
                    zorder=2, label="observed (matched)")
            s = skill(df["obs"], df["mod"])
            ax.set_ylabel(f"{lab} ({unit})")
            ax.set_title(f"{c_}   bias {s['bias']:+.2f} · RMSE {s['rmse']:.2f} "
                         f"· r {s['r']:.2f}  (n={s['n']:.0f})",
                         loc="left", fontsize=9)
        axes[0].legend(ncol=3, fontsize=8, frameon=False, loc="upper right")
        page = f" — page {p + 1} of {npages}" if npages > 1 else ""
        fig.suptitle(f"{lab} at all compared stations, {YEAR}{page}{note}",
                     x=0.008, ha="left", fontsize=12)
        fig.tight_layout(rect=[0, 0, 1, 0.975])
        out = OUT_DIR / (f"fig2_timeseries_{tag or mod_var.lower()}"
                         + (f"_p{p + 1}" if npages > 1 else "") + ".png")
        fig.savefig(out, dpi=150)
        plt.close(fig)
        paths.append(out)
    print(f"{lab}: {len(stns)} stations over {npages} page(s) -> "
          + ", ".join(p_.name for p_ in paths))
    return paths


def scatter_grid(mod_var, obs_col, ncols=5, tag=None, pairs=None, shared=False):
    """One scatter panel per station.

    Panels are scaled INDIVIDUALLY by default. The Wadden wave climate spans
    two orders of magnitude - roughly 0.06 m mean on the flats against 1.4 m
    offshore - so a shared axis collapses the sheltered stations into a corner
    and hides exactly what you want to see. Pass shared=True to compare
    magnitudes directly across panels instead.
    """
    pairs = pairs if pairs is not None else PAIRED.get(mod_var, {})
    if not pairs:
        return None
    # most energetic station first, so the panels read as a gradient
    stns = sorted(pairs, key=lambda c_: -pairs[c_]["obs"].mean())
    unit, lab = UNITS.get(mod_var, ""), LABEL.get(mod_var, mod_var)

    def limits(d):
        lo = float(min(d["obs"].min(), d["mod"].min()))
        hi = float(max(d["obs"].max(), d["mod"].max()))
        pad = 0.04 * (hi - lo) or 0.05
        return [lo - pad, hi + pad]

    glob = limits(pd.concat(pairs.values())) if shared else None
    nrows = int(np.ceil(len(stns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.75 * ncols, 3.0 * nrows),
                             squeeze=False)
    for k, c_ in enumerate(stns):
        ax = axes[k // ncols][k % ncols]
        df = pairs[c_]
        s = skill(df["obs"], df["mod"])
        lim = glob if shared else limits(df)
        ax.plot(lim, lim, color=INK, linewidth=1.0, linestyle="--", zorder=1)
        ax.scatter(df["obs"], df["mod"], s=9, color=C_MOD, alpha=0.45,
                   edgecolor="none", zorder=2)
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
        ax.set_title(f"{c_}\nbias {s['bias']:+.2f} · RMSE {s['rmse']:.2f} · "
                     f"r {s['r']:.2f}", loc="left", fontsize=7.6)
        if k // ncols == nrows - 1:
            ax.set_xlabel(f"observed ({unit})", fontsize=8)
        if k % ncols == 0:
            ax.set_ylabel(f"model ({unit})", fontsize=8)
        ax.tick_params(labelsize=7)
    for k in range(len(stns), nrows * ncols):
        axes[k // ncols][k % ncols].axis("off")
    scale = "shared axes" if shared else "each panel on its own scale"
    fig.suptitle(f"{lab}: model vs observed, every station ({YEAR}). "
                 f"Dashed line is 1:1; {scale}; most energetic first.",
                 x=0.006, ha="left", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    out = OUT_DIR / f"fig3_scatter_{tag or mod_var.lower()}_by_station.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print("->", out.name)
    return out


for mv_, oc_ in PAIRS.items():
    if PAIRED.get(mv_):
        timeseries_pages(mv_, oc_)

In [ ]:
# --- per-station scatter, every station, one figure per variable -----------
for mv_, oc_ in PAIRS.items():
    if PAIRED.get(mv_):
        scatter_grid(mv_, oc_)

In [ ]:
# --- scatter, all stations pooled, one panel per variable -------------------
have = [mv for mv in PAIRS if PAIRED.get(mv)]
fig, axes = plt.subplots(1, len(have), figsize=(4.1 * len(have), 4.2))
for ax, mv in zip(np.atleast_1d(axes), have):
    o = pd.concat([d["obs"] for d in PAIRED[mv].values()])
    m = pd.concat([d["mod"] for d in PAIRED[mv].values()])
    lim = [min(o.min(), m.min()), max(o.max(), m.max())]
    ax.hexbin(o, m, gridsize=42, cmap=SEQ, bins="log", mincnt=1, linewidths=0)
    ax.plot(lim, lim, color=INK, linewidth=1.2, linestyle="--", label="1:1")
    b, a = np.polyfit(o, m, 1)
    xs = np.linspace(*lim, 10)
    ax.plot(xs, b * xs + a, color="#c62f2e", linewidth=1.6,
            label=f"fit, slope {b:.2f}")
    s = skill(o, m)
    unit = {"Hs_out": "m", "Tz_out": "s", "elev": "m"}[mv]
    ax.set_xlabel(f"observed {PAIRS[mv]} ({unit})")
    ax.set_ylabel(f"model {mv} ({unit})")
    ax.set_title(f"{mv}\nbias {s['bias']:+.3f} · RMSE {s['rmse']:.3f} · "
                 f"r {s['r']:.2f}", loc="left", fontsize=9.5)
    ax.legend(fontsize=8, frameon=False, loc="lower right")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig3_scatter_pooled.png", dpi=160)

In [ ]:
# --- Taylor diagram: every station, for Hs ---------------------------------
def taylor(ax, stats, title):
    ax.set_thetamin(0); ax.set_thetamax(90)
    rmax = max(1.6, max(s["std_ratio"] for s in stats.values()) * 1.15)
    for r_ in np.arange(0.5, rmax, 0.5):
        ax.plot(np.linspace(0, np.pi / 2, 90), [r_] * 90, color=GRID, lw=0.6)
    for corr in (0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99):
        th = np.arccos(corr)
        ax.plot([th, th], [0, rmax], color=GRID, lw=0.6)
        ax.text(th, rmax * 1.02, f"{corr}", fontsize=7, color=INK2,
                ha="center")
    # centred-RMSE arcs about the reference point (1, 0)
    th = np.linspace(0, np.pi / 2, 200)
    for crms in (0.25, 0.5, 0.75, 1.0):
        rr = np.cos(th) + np.sqrt(np.maximum(crms ** 2 - np.sin(th) ** 2, 0))
        good = np.sin(th) <= crms
        ax.plot(th[good], rr[good], color="#f3a9a4", lw=0.8, ls=":")
    ax.plot(0, 1, marker="o", ms=9, color=INK, zorder=5)
    ax.text(0.02, 1.0, " reference", fontsize=8, color=INK, va="center")
    # A Taylor diagram already shows correlation (angle) and variance ratio
    # (radius) but says nothing about bias, so colour carries the bias.
    names = sorted(stats, key=lambda k: -stats[k]["r"])
    blim = max(abs(stats[n_]["bias"]) for n_ in names) or 0.1
    pts = ax.scatter(
        [np.arccos(np.clip(stats[n_]["r"], -1, 1)) for n_ in names],
        [stats[n_]["std_ratio"] for n_ in names],
        c=[stats[n_]["bias"] for n_ in names], cmap=DIV,
        norm=TwoSlopeNorm(0, -blim, blim), s=90, edgecolor="white",
        linewidth=1.2, zorder=4)
    for k_, n_ in enumerate(names, 1):
        ax.annotate(str(k_), (np.arccos(np.clip(stats[n_]["r"], -1, 1)),
                              stats[n_]["std_ratio"]),
                    fontsize=6.5, color=INK, ha="center", va="center",
                    zorder=5)
    ax.set_rmax(rmax)
    ax.set_title(title, loc="left", pad=18)
    return pts, names

stats_hs = {c_: skill(d["obs"], d["mod"]) for c_, d in PAIRED["Hs_out"].items()}
stats_hs = {k: v for k, v in stats_hs.items() if np.isfinite(v["r"])}
fig = plt.figure(figsize=(10.5, 6.6))
ax = fig.add_subplot(111, projection="polar")
pts, names = taylor(ax, stats_hs,
                    f"Taylor diagram — Hs, {YEAR}   "
                    "(angle = correlation, radius = std ratio, colour = bias)")
cb = fig.colorbar(pts, ax=ax, pad=0.10, fraction=0.035)
cb.set_label("bias, model − observed (m)"); cb.outline.set_visible(False)
legend_txt = chr(10).join(f"{k}. {n}" for k, n in enumerate(names, 1))
fig.text(0.72, 0.88, legend_txt,
         fontsize=7, va="top", color=INK2)
fig.tight_layout(); fig.savefig(OUT_DIR / "fig4_taylor_hs.png", dpi=160)

In [ ]:
# --- map of Hs bias --------------------------------------------------------
b = pd.Series({c_: s["bias"] for c_, s in stats_hs.items()})
lim = float(np.nanmax(np.abs(b))) or 0.1
fig, ax = plt.subplots(figsize=(11.5, 5.4))
ax.pcolormesh(lon2d, lat2d, np.where(wet, 1.0, np.nan),
              cmap=LinearSegmentedColormap.from_list("g", ["#eceae4", "#eceae4"]),
              shading="auto", zorder=0)
sc = ax.scatter(match.loc[b.index, "lon"], match.loc[b.index, "lat"],
                c=b.values, cmap=DIV, norm=TwoSlopeNorm(0, -lim, lim),
                s=125, edgecolor="white", linewidth=1.5, zorder=3)
for c_ in b.index:
    ax.annotate(f"{b[c_]:+.2f}", (match.loc[c_, "lon"], match.loc[c_, "lat"]),
                fontsize=6.4, color=INK2, xytext=(0, 8),
                textcoords="offset points", ha="center")
cb = fig.colorbar(sc, ax=ax, pad=0.012, fraction=0.03)
cb.set_label("Hs bias, model − observed (m)"); cb.outline.set_visible(False)
ax.set_aspect(1 / np.cos(np.deg2rad(lat0)))
ax.set_xlabel("longitude (°E)"); ax.set_ylabel("latitude (°N)")
ax.set_title(f"Hs bias by station, {YEAR}", loc="left")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig5_bias_map.png", dpi=160)

In [ ]:
# --- conditional bias: does skill depend on sea state? ---------------------
o = pd.concat([d["obs"] for d in PAIRED["Hs_out"].values()])
m = pd.concat([d["mod"] for d in PAIRED["Hs_out"].values()])
edges = np.array([0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 10.0])
lab, bias, rmse, n = [], [], [], []
for k in range(len(edges) - 1):
    sel_ = (o >= edges[k]) & (o < edges[k + 1])
    if sel_.sum() < 10:
        continue
    lab.append(f"{edges[k]:g}–{edges[k+1]:g}")
    bias.append((m[sel_] - o[sel_]).mean())
    rmse.append(np.sqrt(((m[sel_] - o[sel_]) ** 2).mean()))
    n.append(int(sel_.sum()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.axhline(0, color=INK, lw=1)
ax1.bar(lab, bias, color=C_MOD, width=0.66)
ax1.set_ylabel("bias (m)"); ax1.set_xlabel("observed Hs bin (m)")
ax1.set_title("Bias by sea state", loc="left")
for x_, v_, nn in zip(lab, bias, n):
    ax1.annotate(f"n={nn}", (x_, v_), fontsize=6.6, ha="center", color=INK2,
                 xytext=(0, 4 if v_ >= 0 else -11), textcoords="offset points")
ax2.bar(lab, rmse, color=C_MOD, width=0.66)
ax2.set_ylabel("RMSE (m)"); ax2.set_xlabel("observed Hs bin (m)")
ax2.set_title("RMSE by sea state", loc="left")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig6_conditional.png", dpi=160)

---
## 9. Water level — aliased, read with care

Daily 00:00 samples alias M2 to ~14.8 days (see the header). The comparison
below is therefore **not** a measure of tidal skill. What it *can* still show:

- a datum offset between GETM's reference level and NAP (the bias term),
- gross problems in the surge/residual signal,
- whether the model dries a cell that the observations show as wet.

For real tidal validation you need sub-hourly model output and a harmonic
analysis (e.g. `utide`) of both series.

In [ ]:
if PAIRED.get("elev"):
    t = tables["elev"]
    print("Water level, daily 00:00 snapshots (ALIASED — see the note above)\n")
    display(t[["n", "obs_mean", "mod_mean", "bias", "rmse", "r", "std_ratio"]].round(3))

    # the time series themselves are already plotted for every station by the
    # section-8 loop (fig2_timeseries_elev*). Here: the difference for every
    # station, where the M2 alias is unmistakable.
    stns = sorted(PAIRED["elev"], key=lambda k: -len(PAIRED["elev"][k]))
    fig, axes = plt.subplots(len(stns), 1, figsize=(13, 1.5 * len(stns)),
                             sharex=True)
    for ax, c_ in zip(np.atleast_1d(axes), stns):
        df = PAIRED["elev"][c_]
        # on the full model axis so gaps break the line rather than being
        # bridged by a straight segment
        diff = (df["mod"] - df["obs"]).reindex(mtimes)
        ax.plot(diff.index, diff.values, color="#c62f2e", lw=1.0)
        ax.axhline(0, color=INK, lw=1)
        ax.set_ylabel("model − obs (m)")
        ax.set_title(f"{c_}   mean {float((df['mod']-df['obs']).mean()):+.3f} m",
                     loc="left", fontsize=9)
    fig.suptitle("Water level difference at 00:00 — the ~15-day beat is the "
                 "M2 alias, not model drift", x=0.008, ha="left", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.975])
    fig.savefig(OUT_DIR / "fig8_elev_difference.png", dpi=150)
    plt.close(fig)
    print(f"difference panels for {len(stns)} stations -> "
          "fig8_elev_difference.png")
else:
    print("no water-level pairs")

---
## 10. Currents — ready, but empty for 2015

No RWS current observations exist in the Dutch Wadden Sea in 2015. This cell
runs unchanged for a year ≥ 2020 (Eemshaven from 2020, Den Helder from 2025).

Before trusting the result, settle two things:

1. **Component convention.** GETM's `u`/`v` are documented as *"velocity in
   global x/y-direction"*, i.e. already rotated to geographic east/north, so no
   `convc` rotation should be applied. Verify against your setup — the grid
   carries a non-zero `convc`.
2. **What the observation represents.** The Eemshaven bin coordinate is *not* a
   depth (README §4): the bin-averaged `u`/`v` may be a **cross-channel** mean,
   in which case it should be compared against a GETM transect average rather
   than the single nearest cell.

In [ ]:
cur_stn = [c_ for c_ in STN
           if {"u", "v"} <= set(OBS[c_].columns) and OBS[c_]["u"].notna().any()]
if not cur_stn:
    print(f"No current observations in {YEAR} — nothing to compare. "
          "Re-run with YEAR >= 2020.")
else:
    rows = []
    for c_ in cur_stn:
        how = "instantaneous"          # u/v carry no `averaged` attribute
        for comp in ("u", "v"):
            o = obs_at(OBS[c_][comp], mtimes, how)
            m = pd.Series(MOD[comp].sel(station=c_).values, index=mtimes)
            d = pd.DataFrame({"obs": o, "mod": m}).dropna()
            if len(d) >= MIN_PAIRS:
                rows.append({"code": c_, "component": comp,
                             **skill(d["obs"], d["mod"])})
    if rows:
        display(pd.DataFrame(rows).set_index(["code", "component"]).round(3))
    print("\nReminder: daily 00:00 samples alias the tidal currents exactly as "
          "they alias the water level.")

---
## 11. Wind forcing sanity check

Wave errors usually trace back to the wind. This compares the model's own
forcing (`EUWIND`/`EVWIND`) against the nearest KNMI station, which is a check
on the **forcing**, not on the model.

In [ ]:
kn_meta = pd.read_csv(OBS_ROOT / "processed" / "knmi_wind_stations.csv")
rows = []
for _, k in kn_meta.iterrows():
    kx, ky = _xy(float(k["lon"]), float(k["lat"]))
    dist, idx = tree.query([kx, ky])
    if dist > 15:                     # KNMI stations are on land; allow slack
        continue
    j, i = int(jw[idx]), int(iw[idx])
    w = pd.read_csv(OBS_ROOT / "processed" / "knmi_wind" / f"knmi_{int(k['stn'])}.csv.gz",
                    index_col=0, parse_dates=True).loc[f"{YEAR}-01-01":f"{YEAR}-12-31"]
    if w.empty:
        continue
    mu, mv = [], []
    for f in files:
        with xr.open_dataset(f) as d:
            if "EUWIND" not in d.variables:
                break
            mu.append(d["EUWIND"].isel(yc=j, xc=i).to_series())
            mv.append(d["EVWIND"].isel(yc=j, xc=i).to_series())
    if not mu:
        print("model has no EUWIND/EVWIND"); break
    mu = pd.concat(mu); mv = pd.concat(mv)
    mu, mv = mu[~mu.index.duplicated()], mv[~mv.index.duplicated()]
    how = OBS_CONVENTION.get("Hs_out", "instantaneous")
    ou = obs_at(w["wind_u"], mu.index, how)
    ov_ = obs_at(w["wind_v"], mv.index, how)
    spd_o = np.hypot(ou, ov_); spd_m = np.hypot(mu, mv)
    d = pd.DataFrame({"obs": spd_o, "mod": spd_m}).dropna()
    if len(d) >= MIN_PAIRS:
        rows.append({"knmi": f"{int(k['stn'])} {k['label']}",
                     "dist_km": round(float(dist), 1), **skill(d["obs"], d["mod"])})

if rows:
    display(pd.DataFrame(rows).set_index("knmi")[
        ["dist_km", "n", "obs_mean", "mod_mean", "bias", "rmse", "r"]].round(3))
    print("\nA systematic wind-speed bias here will propagate into Hs roughly "
          "as Hs ~ U^2 in fetch-limited conditions.")
else:
    print("no KNMI/model wind pairs")

---
## 12. Export

In [ ]:
# CSV first: it needs nothing beyond pandas, so it always succeeds.
match.to_csv(OUT_DIR / f"station_matching_{YEAR}.csv")
conv.to_csv(OUT_DIR / f"sampling_convention_{YEAR}.csv")
for mv, t in tables.items():
    t.round(4).to_csv(OUT_DIR / f"skill_{mv}_{YEAR}.csv")

# One tidy long-form table, convenient for plotting or pasting into a paper.
long = []
for mv, t in tables.items():
    for stn, row in t.iterrows():
        long.append({"variable": mv, "observation": PAIRS.get(mv, ""),
                     "station": stn, **row.to_dict()})
long = pd.DataFrame(long)
long.round(4).to_csv(OUT_DIR / f"skill_all_{YEAR}.csv", index=False)
print(f"skill_all_{YEAR}.csv: {len(long)} rows "
      f"({long['variable'].nunique()} variables)")

# Excel is optional - openpyxl is often missing on a cluster environment.
try:
    with pd.ExcelWriter(OUT_DIR / f"validation_summary_{YEAR}.xlsx") as xl:
        match.to_excel(xl, sheet_name="station_matching")
        conv.to_excel(xl, sheet_name="sampling_convention")
        for mv, t in tables.items():
            t.round(4).to_excel(xl, sheet_name=mv[:31])
    print("wrote validation_summary.xlsx")
except ImportError as exc:
    print(f"skipped the .xlsx ({exc}). The CSVs above hold the same content; "
          "`pip install openpyxl` if you want the workbook.")
except Exception as exc:
    print(f"skipped the .xlsx ({type(exc).__name__}: {exc})")

print("\nwritten to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.iterdir()):
    kb = f.stat().st_size / 1024
    print(f"   {f.name:<46} "
          + (f"{kb:>8.1f} kB" if kb < 100 else f"{kb:>8.0f} kB"))

---
## 13. Reading the results

**Headline numbers to quote.** For each variable: bias, RMSE, scatter index
and correlation from the `** ALL **` row. Scatter index is the most comparable
across studies; for Hs in coastal models, SI below ~0.3 is respectable.

**Diagnosing a bias.**

| Symptom | Likely cause |
|---|---|
| `std_ratio` < 1 and `slope` < 1 everywhere | model under-responds — check wind forcing (§11) or whitecapping |
| Bias grows with Hs (§8, conditional plot) | dissipation/growth calibration, not a mean offset |
| Bias only at flat stations, fine offshore | depth-limited breaking or bathymetry, not the wave physics |
| `elev` bias constant across all stations | datum offset between GETM reference level and NAP — subtract it |

**Before drawing conclusions**, confirm:

- **Time zone.** RWS observations here are UTC. GETM `time` is assumed UTC. A
  one-hour error would show up as a small phase error in `elev`; a MET/UTC mix
  is the most common silent bug in Dutch model validation.
- **The sampling convention** decided in §6 — it is inferred from the data, not
  from documentation.
- **Nearest-cell matching** at 500 m resolution. A station in a narrow channel
  may be matched to a cell that is not hydrodynamically equivalent. `dist_km`
  and `model_depth_m` in §4 are the first place to look if one station is an
  outlier; compare `model_depth_m` against `bed_level_nap_m` in
  `../catalog/station_depth.csv`.

**To extend this**, the cheapest wins in order: daily output → hourly output
(unlocks real tidal validation and removes the aliasing caveat entirely);
2015 → multiple years (storm statistics); add a station-by-station QQ plot for
the extremes, which is where a wave model usually fails first.